In [6]:
Batch compute layer

Most common:

Apache Spark
large-scale feature computation
joins
aggregations
historical windows
training data generation

Examples:

user_watch_history
        |
        Spark job
        |
        +--> user_genre_affinity_30d
        +--> avg_watch_time_30d
        +--> completion_rate_30d
        +--> click_rate_7d

Alternative:

Apache Flink batch/stream hybrid (less common for pure offline)
Apache Beam
Hive/Trino/Snowflake/BigQuery-style SQL engines
Feature Store

The real architecture usually separates:

Offline Feature Store
        |
        |
        +---- Training
        |
        |
        +---- Materialization job
                    |
                    v
              Online Feature Store

Examples of real feature stores:

Feast
Tecton

Your current:

OfflineFeatureStore

should become conceptually:

SparkFeaturePipeline
        |
        v
OfflineFeatureStore
        |
        v
FeatureMaterializationJob
        |
        v
RedisOnlineFeatureStore
For our simulator, I would add:
SparkSimulator
    |
    | consumes
    |
DataLake
    |
    |
FeatureEngineeringJobs
    |
    |
OfflineFeatureStore
    |
    |
TrainingPipeline

Then later add:

FlinkSimulator
    |
Kafka/EventBus
    |
OnlineFeaturePipeline
    |
Redis

So the final realistic system becomes:

                 Events
                    |
                    v
                 Kafka
              /          \
             /            \
        Flink              Data Lake
          |                    |
          v                    v
       Redis             Spark Batch
          |                    |
          |                    v
          |             Offline Feature Store
          |                    |
          +-----------> Training Pipeline
                               |
                               v
                         Two Tower / Ranker
                               |
                               v
                              HNSW
                               |
                               v
                         Online Retrieval

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 41)

In [7]:
What is still missing before calling it a more realistic ML system:

Training actually consumes feedback
Verify the training dataset contains meaningful (user, item, label) examples.
Right now this is the biggest thing to validate.
Model lifecycle
Train new model
Generate new embeddings
Rebuild/swap HNSW index
Serve new version
Evaluation
Track metrics:
CTR
watch completion
recall@K
ranking metrics
Incremental behavior
New users/features entering the system
Periodic retraining
Updating embeddings/index
Component boundaries
You already planned this:
swap TwoTower
swap feature store
swap retrieval
later swap Kafka/Flink/Redis simulations

SyntaxError: invalid syntax (2048715761.py, line 1)

In [8]:
import random
from abc import ABC, abstractmethod
from dataclasses import dataclass

import math

In [11]:
"""

!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip

--2026-08-31 07:00:55--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
ERROR: cannot verify files.grouplens.org's certificate, issued by ‘CN=InCommon ECC Server CA 2,O=Internet2,C=US’:
  Issued certificate has expired.
To connect to files.grouplens.org insecurely, use `--no-check-certificate'.
unzip:  cannot find or open ml-100k.zip, ml-100k.zip.zip or ml-100k.zip.ZIP.


In [13]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

--2026-08-31 07:02:03--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
  Issued certificate has expired.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  16.1MB/s    in 0.3s    

2026-08-31 07:02:03 (16.1 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test  

In [14]:
!ls ml-100k

allbut.pl  u1.base  u2.test  u4.base  u5.test  ub.base	u.genre  u.occupation
mku.sh	   u1.test  u3.base  u4.test  ua.base  ub.test	u.info	 u.user
README	   u2.base  u3.test  u5.base  ua.test  u.data	u.item


In [3]:
#helpers

def cosine_distance(a, b):

    dot = sum(
        x * y
        for x, y in zip(a, b)
    )

    norm_a = sum(
        x * x
        for x in a
    ) ** 0.5

    norm_b = sum(
        y * y
        for y in b
    ) ** 0.5

    return 1 - (dot / (norm_a * norm_b))

In [15]:
import pandas as pd

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

users = pd.read_csv(
    "ml-100k/u.user",
    sep="|",
    names=["user_id", "age", "gender", "occupation", "zip_code"]
)



In [16]:
from dataclasses import dataclass

@dataclass
class User:
    user_id: int
    age: int
    gender: str
    occupation: str
    zip_code: str


@dataclass
class Movie:
    movie_id: int
    title: str
    release_date: str
    genres: list[str]


@dataclass
class Rating:
    user_id: int
    movie_id: int
    rating: int
    timestamp: int

In [17]:
genre_names = [
    "unknown", "Action", "Adventure", "Animation", "Children's",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]

In [18]:
user_objects = {
    row.user_id: User(
        row.user_id,
        row.age,
        row.gender,
        row.occupation,
        row.zip_code,
    )
    for row in users.itertuples(index=False)
}

movie_objects = {}

for row in movies.itertuples(index=False):
    genres = [
        genre
        for genre, flag in zip(genre_names, row[5:])
        if flag == 1
    ]

    movie_objects[row[0]] = Movie(
        movie_id=row[0],
        title=row[1],
        release_date=row[2],
        genres=genres,
    )

rating_objects = [
Rating(*row)
for row in ratings.itertuples(index=False, name=None)
]

In [ ]:
len(user_objects)

In [19]:
class UserState: #bottom

    def __init__(self, user):

        self.user_id = user.user_id

        # Ground truth
        self.ratings = {}
        self.watched_movies = set()

        # Derived from history
        self.genre_preferences = {}
        self.average_rating = 0.0
        self.activity_level = 0.0

        # Simulation
        self.session_count = 0
        self.last_active = None

class MovieState:

  def __init__(self, movie):

      self.movie_id = movie.movie_id

      # Static
      self.title = movie.title
      self.genres = movie.genres

      # Dynamic
      self.impressions = 0
      self.clicks = 0
      self.watches = 0
      self.ratings = []

      self.num_ratings = 0
      self.average_rating = 0.0
      self.popularity = 0
      self.trending_score = 0

In [20]:
from collections import defaultdict

def initialize_user_states(
    user_states,
    movie_objects,
    rating_objects
):

    for rating in rating_objects:

        state = user_states[rating.user_id]

        state.watched_movies.add(rating.movie_id)
        state.ratings[rating.movie_id] = rating.rating

        movie = movie_objects[rating.movie_id]

        for genre in movie.genres:
            state.genre_preferences[genre] = (
                state.genre_preferences.get(genre, 0) + rating.rating
            )

    for state in user_states.values():

        if state.ratings:
            state.average_rating = (
                sum(state.ratings.values()) /
                len(state.ratings)
            )

        state.activity_level = len(state.ratings)

In [21]:
user_states = {}

for user_id, user in user_objects.items():
    user_states[user_id] = UserState(user)

initialize_user_states(
    user_states,
    movie_objects,
    rating_objects
)

state = user_states[1]

print(state.average_rating)
print(state.activity_level)
print(state.genre_preferences)
print(len(state.watched_movies))

3.610294117647059
272
{'Drama': 420, 'Animation': 40, 'Comedy': 316, 'Action': 250, 'Romance': 173, 'Thriller': 188, 'Sci-Fi': 172, 'Musical': 38, 'Adventure': 123, 'Crime': 86, 'Horror': 45, 'War': 92, "Children's": 55, 'Western': 22, 'Mystery': 18, 'Fantasy': 7, 'unknown': 4, 'Film-Noir': 5, 'Documentary': 24}
272


In [22]:
movie_states = {
    movie_id: MovieState(movie)
    for movie_id, movie in movie_objects.items()
}

In [ ]:
len(user_states), len(movie_states)

(943, 1682)

In [23]:
from collections import defaultdict

movie_rating_sum = defaultdict(float)

for r in rating_objects:
    user_states[r.user_id].watched_movies.add(r.movie_id)

    movie_states[r.movie_id].num_ratings += 1
    movie_rating_sum[r.movie_id] += r.rating

for movie_id, state in movie_states.items():
    if state.num_ratings > 0:
        state.average_rating = (
            movie_rating_sum[movie_id] / state.num_ratings
        )

In [24]:
@dataclass
class Event:
    timestamp: int
    user_id: int

@dataclass
class RecommendationShownEvent(Event):
    recommendation_id: int
    movie_ids: list[int]


@dataclass
class ClickEvent(Event):
    movie_id: int

@dataclass
class RecommendationClickedEvent(Event):
    recommendation_id: int
    movie_id: int


@dataclass
class WatchEvent(Event):
    movie_id: int
    watch_time: float      # seconds
    completion: float      # 0-1


@dataclass
class RatingEvent(Event):
    movie_id: int
    rating: float

@dataclass
class Session:
    session_id: int
    user_id: int
    started_at: int

@dataclass
class RetrievalRequest:
    user_id: int
    k: int

In [25]:
class Clock:

    def __init__(self):
        self.tick_number = 0

    def tick(self):
        self.tick_number += 1

    @property
    def now(self):
        return self.tick_number

In [ ]:
Simulator karnel responsibilities:

Time (Clock)
Service lifecycle (start, stop, tick)
Scheduling (who runs when)
Failure injection
Simulation configuration
Global observability

Everything else belongs to services.

For example:

Kafka owns topics.
Airflow owns DAGs.
Spark owns jobs.
Flink owns operators.
World owns users.

"The Simulator is to the simulated ML platform what an operating system is to processes."

In [26]:
class Simulator:

    def __init__(self):
        self.clock = Clock()

        # All services running in the simulation
        self.services = {}

        # Global event bus (NOT Kafka)
        self.event_bus = []

        # Global configuration
        self.config = {}

        # Failure injection
        self.scenarios = []

        # Metrics about the simulation itself
        self.metrics = {}

        # Randomness
        self.random = random.Random(42)

        self.running = False

    #def register(self, service):
      #  self.services[service.name] = service

    def register(self, service):
        self.services[service.name] = service
        service.start(self)

    def unregister(self, name):
        del self.services[name]

    #def publish(self, event):
       # self.event_bus.append(event)
    """
    def publish(self, event):
        print(event)
        self.event_bus.append(event)
    """
    def publish(self, event):

        print(event)

        kafka = self.get_service("Kafka")

        kafka.receive(event)


    def get_service(self, name):
        return self.services[name]

    def tick(self):

        self.clock.tick()

        for service in self.services.values():
            service.tick(self)

    def run(self, ticks=None):

        self.running = True

        while self.running:

            self.tick()

            if ticks is not None:
                ticks -= 1
                if ticks == 0:
                    break

    def stop(self):
        self.running = False

In [27]:
from enum import Enum


class ServiceStatus(Enum):
    STOPPED = "STOPPED"
    STARTING = "STARTING"
    RUNNING = "RUNNING"
    DEGRADED = "DEGRADED"
    FAILED = "FAILED"


class Service:

    def __init__(self, name=None):
        self.name = name or self.__class__.__name__
        self.status = ServiceStatus.STOPPED

        self.inbox = []
        self.outbox = []

        self.metrics = {}
        self.logs = []
        self.config = {}

    def start(self, sim):
        self.status = ServiceStatus.RUNNING

    def stop(self, sim):
        self.status = ServiceStatus.STOPPED

    def receive(self, event):
        self.inbox.append(event)

    def process(self, sim):
        """Override in subclasses."""
        pass

    def emit(self):
        events = self.outbox
        self.outbox = []
        return events

    def tick(self, sim):
        if self.status != ServiceStatus.RUNNING:
            return

        self.process(sim)

        for event in self.emit():
            sim.publish(event)

    def log(self, message):
        self.logs.append(message)

# Services:

In [28]:
class Kafka(Service):

    def __init__(self):
        super().__init__()
        self.topics = {}
        self.offsets = {}
        self.subscribers = {}

    def receive(self, event):

        topic = type(event).__name__

        if topic not in self.topics:
            self.topics[topic] = []

        self.topics[topic].append(event)

    def subscribe(self, consumer_name, topic):
        self.subscribers.setdefault(topic, []).append(consumer_name)
        self.offsets[(consumer_name, topic)] = 0


    def poll(self, consumer_name, topic):

        offset = self.offsets[(consumer_name, topic)]

        messages = self.topics.get(topic, [])[offset:]

        self.offsets[(consumer_name, topic)] += len(messages)

        return messages

class Airflow(Service):
    pass

class Spark(Service):
    pass


class Redis(Service):

    def __init__(self):
        super().__init__()
        self.store = {}

    def receive(self, feature):

        if "user_id" in feature:
            key = (
                feature["feature"],
                feature["user_id"]
            )

        elif "movie_id" in feature:
            key = (
                feature["feature"],
                feature["movie_id"]
            )

        self.store[key] = feature["value"]

    def get(self, feature, entity_id):
        return self.store.get(
            (feature, entity_id)
        )

    def set(self, feature, entity_id, value):
        self.store[(feature, entity_id)] = value

class Feast(Service):
    pass

class MLflow(Service):
    pass

class DataLake(Service):

    def __init__(self):
        super().__init__()
        self.events = []

    def receive(self, event):
        self.events.append(event)

    def start(self, sim):
        super().start(sim)

        kafka = sim.get_service("Kafka")

        kafka.subscribe(
            "DataLake",
            "RecommendationClickedEvent"
        )

        kafka.subscribe(
            "DataLake",
            "WatchEvent"
        )

        kafka.subscribe(
            "DataLake",
            "RatingEvent"
        )

    def process(self, sim):

        kafka = sim.get_service("Kafka")

        for topic in [
            "RecommendationClickedEvent",
            "WatchEvent",
            "RatingEvent"
        ]:
            events = kafka.poll(
                "DataLake",
                topic
            )

            for event in events:
                self.events.append(event)

class RetrievalService(Service, ABC):

    @abstractmethod
    def retrieve(self, request: RetrievalRequest):
        pass


class ModelServer(Service):

  def __init__(
      self,
      retriever,
      ranker=None,
      reranker=None
  ):
      super().__init__()

      self.retriever = retriever
      self.ranker = ranker
      self.reranker = reranker

  def recommend(self, user):

      request = RetrievalRequest(
          user_id=user.user_id,
          k=100
      )

      candidates = self.retriever.retrieve(request)

      if self.ranker is not None:
          candidates = self.ranker.rank(
              user,
              candidates
          )

      if self.reranker is not None:
          candidates = self.reranker.rerank(
              user,
              candidates
          )

      return candidates

class Monitor(Service):
    pass

In [29]:
class Flink(Service):

    def __init__(self):
        super().__init__()

        self.consumer_name = "Flink"

        self.topics = [
            "RecommendationClickedEvent",
            "WatchEvent",
            "RatingEvent"
        ]

        self.state = {
            "activity_level": {},
            "item_popularity": {}
        }

        self.output = []

        # Shared feature computation
        self.rating_features = RatingFeatureComputer()
        self.item_rating_features = ItemRatingFeatureComputer()

    def start(self, sim):
        super().start(sim)

        kafka = sim.get_service("Kafka")

        for topic in self.topics:
            kafka.subscribe(
                self.consumer_name,
                topic
            )

    def process(self, sim):

        kafka = sim.get_service("Kafka")

        for topic in self.topics:

            events = kafka.poll(
                self.consumer_name,
                topic
            )

            for event in events:

                if isinstance(event, RecommendationClickedEvent):
                    self.handle_recommendation_clicked(sim, event)

                elif isinstance(event, WatchEvent):
                    self.handle_watch(sim, event)

                elif isinstance(event, RatingEvent):
                    self.handle_rating(sim, event)


    def update_activity_level(self, sim, user_id):

        self.state["activity_level"][user_id] = (
            self.state["activity_level"].get(user_id, 0) + 1
        )

        redis = sim.get_service("Redis")

        redis.receive({
            "feature": "activity_level",
            "user_id": user_id,
            "value": self.state["activity_level"][user_id]
        })


    def update_item_popularity(self, sim, movie_id):

        self.state["item_popularity"][movie_id] = (
            self.state["item_popularity"].get(movie_id, 0) + 1
        )

        redis = sim.get_service("Redis")

        redis.receive({
            "feature": "item_popularity",
            "movie_id": movie_id,
            "value": self.state["item_popularity"][movie_id]
        })


    def handle_recommendation_clicked(self, sim, event):

        user_id = event.user_id

        self.update_activity_level(
            sim,
            user_id
        )

        self.state.setdefault("user_click_counts", {})

        self.state["user_click_counts"][user_id] = (
            self.state["user_click_counts"].get(user_id, 0) + 1
        )

        redis = sim.get_service("Redis")

        redis.receive({
            "feature": "user_click_counts",
            "user_id": user_id,
            "value": self.state["user_click_counts"][user_id]
        })


    def handle_rating(self, sim, event):

        self.update_activity_level(
            sim,
            event.user_id
        )

        redis = sim.get_service("Redis")

        # User average rating
        average_rating = self.rating_features.update(
            event.user_id,
            event.rating
        )

        redis.receive({
            "feature": "average_rating",
            "user_id": event.user_id,
            "value": average_rating
        })


        # Item average rating
        item_average_rating = self.item_rating_features.update(
            event.movie_id,
            event.rating
        )

        redis.receive({
            "feature": "item_average_rating",
            "movie_id": event.movie_id,
            "value": item_average_rating
        })

    def handle_watch(self, sim, event):

        user_id = event.user_id
        movie_id = event.movie_id

        redis = sim.get_service("Redis")


        # -------------------------
        # Activity
        # -------------------------

        self.update_activity_level(
            sim,
            user_id
        )


        # -------------------------
        # Item popularity
        # -------------------------

        self.update_item_popularity(
            sim,
            movie_id
        )


        # -------------------------
        # Watch count
        # -------------------------

        self.state.setdefault(
            "watch_count",
            {}
        )

        self.state["watch_count"][user_id] = (
            self.state["watch_count"].get(user_id, 0) + 1
        )

        redis.receive({
            "feature": "watch_count",
            "user_id": user_id,
            "value": self.state["watch_count"][user_id]
        })


        # -------------------------
        # Genre preference update
        # -------------------------

        world = sim.get_service("World")
        movie = world.movie_objects[movie_id]


        self.state.setdefault(
            "genre_counts",
            {}
        )


        # New users start with empty online state
        if user_id not in self.state["genre_counts"]:

            self.state["genre_counts"][user_id] = {}


        counts = self.state["genre_counts"][user_id]


        # Add new watch signal
        for genre in movie.genres:

            counts[genre] = (
                counts.get(genre, 0) + 1
            )


        total = sum(
            counts.values()
        )


        if total > 0:

            genre_preferences = {
                genre: count / total
                for genre, count in counts.items()
            }


            redis.receive({
                "feature": "genre_preferences",
                "user_id": user_id,
                "value": genre_preferences
            })


            print(
                "FINAL GENRE PREF",
                user_id,
                genre_preferences
            )




# World:

In [30]:
from uuid import uuid4


class World(Service):

    def __init__(
        self,
        user_objects,
        movie_objects,
        user_states,
        movie_states
    ):
        super().__init__()

        self.user_objects = user_objects
        self.movie_objects = movie_objects

        self.user_states = user_states
        self.movie_states = movie_states

        self.active_sessions = {}

        self.user_simulator = UserSimulator(self)


    def process(self, sim):
        print("World processing")

        active_users = self.sample_active_users(sim)

        for user in active_users:
            self.simulate_user(user, sim)


    def sample_active_users(self, sim):
        # Decide which users are active this tick.
        # For now, activate one random user.

        return [sim.random.choice(list(self.user_objects.values()))]


    def simulate_user(self, user, sim):

        model_server = sim.get_service("ModelServer")

        movie_ids = model_server.recommend(user)

        events = self.user_simulator.react(
            sim=sim,
            user=user,
            recommendation_id=sim.clock.now,
            movie_ids=movie_ids
        )

        self.outbox.extend(events)

    def process_events(self, events):

        for event in events:

            self.process_event(event)

            self.outbox.append(event)


    def process_event(self, event):

        if isinstance(event, RecommendationShownEvent):
            self.process_shown(event)

        elif isinstance(event, RecommendationClickedEvent):
            self.process_click(event)

        elif isinstance(event, WatchEvent):
            self.process_watch(event)

        elif isinstance(event, RatingEvent):
            self.process_rating(event)


    def process_shown(self, event):

        movie_state = self.movie_states[event.movie_ids[0]]  # Placeholder

        movie_state.impressions += 1


    def process_click(self, event):

        user_state = self.user_states[event.user_id]
        movie_state = self.movie_states[event.movie_id]

        user_state.click_count += 1
        movie_state.click_count += 1


    def process_watch(self, event):

        user_state = self.user_states[event.user_id]
        movie_state = self.movie_states[event.movie_id]

        user_state.watch_count += 1
        movie_state.watch_count += 1


    def process_rating(self, event):

        user_state = self.user_states[event.user_id]
        movie_state = self.movie_states[event.movie_id]

        user_state.rating_count += 1

        user_state.ratings[event.movie_id] = event.rating

        movie_state.ratings.append(event.rating)
        movie_state.average_rating = (
            sum(movie_state.ratings) / len(movie_state.ratings)
        )

In [31]:
class UserSimulator:

    def __init__(self, world):
        self.world = world
        #self.click_model = RandomClickModel()
        #self.watch_model = RandomWatchModel()
        #self.rating_model = RandomRatingModel()

    def react(
        self,
        sim,
        user,
        recommendation_id,
        movie_ids
    ):

        events = []

        events.append(
            self.show(
                sim,
                user,
                recommendation_id,
                movie_ids
            )
        )

        #if self.should_click():
        if self.should_click(user, movie_ids):

            clicked = self.click(
                sim,
                user,
                recommendation_id,
                movie_ids
            )

            events.append(clicked)

            if self.should_watch():

                events.append(
                    self.watch(
                        sim,
                        user,
                        clicked.movie_id
                    )
                )

                if self.should_rate():

                    events.append(
                        self.rate(
                            sim,
                            user,
                            clicked.movie_id
                        )
                    )

        return events


    def show(
        self,
        sim,
        user,
        recommendation_id,
        movie_ids
    ):

        return RecommendationShownEvent(
            timestamp=sim.clock.now,
            user_id=user.user_id,
            recommendation_id=recommendation_id,
            movie_ids=movie_ids
        )


    #def should_click(self):
        #return random.random() < 0.2

    def should_click(self, user, movie_ids):

        user_state = self.world.user_states[user.user_id]

        for movie_id in movie_ids:

            movie = self.world.movie_objects[movie_id]

            if any(
                genre in user_state.genre_preferences
                for genre in movie.genres
            ):
                return random.random() < 0.6

        return random.random() < 0.1


    def click(
        self,
        sim,
        user,
        recommendation_id,
        movie_ids
    ):

        user_state = self.world.user_states[user.user_id]

        best_movie = None
        best_score = -1

        for movie_id in movie_ids:

            movie = self.world.movie_objects[movie_id]

            score = sum(
                user_state.genre_preferences.get(genre, 0)
                for genre in movie.genres
            )

            if score > best_score:
                best_score = score
                best_movie = movie_id

        print(f"User {user.user_id} clicked {best_movie} (score={best_score})")

        return RecommendationClickedEvent(
            timestamp=sim.clock.now,
            user_id=user.user_id,
            recommendation_id=recommendation_id,
            movie_id=best_movie
        )


    def should_watch(self):
        return random.random() < 0.8


    def watch(
        self,
        sim,
        user,
        movie_id
    ):

        return WatchEvent(
            timestamp=sim.clock.now,
            user_id=user.user_id,
            movie_id=movie_id,
            watch_time=random.uniform(5, 120),
            completion=random.uniform(0.1, 1.0)
        )


    def should_rate(self):
        return random.random() < 0.1


    def rate(
        self,
        sim,
        user,
        movie_id
    ):

        return RatingEvent(
            timestamp=sim.clock.now,
            user_id=user.user_id,
            movie_id=movie_id,
            rating=random.randint(1, 5)
        )

# Feature computation

In [32]:
class RatingFeatureComputer:

    def __init__(self):
        self.rating_sum = {}
        self.rating_count = {}

    def update(self, entity_id, rating):

        self.rating_sum[entity_id] = (
            self.rating_sum.get(entity_id, 0) + rating
        )

        self.rating_count[entity_id] = (
            self.rating_count.get(entity_id, 0) + 1
        )

        return (
            self.rating_sum[entity_id] /
            self.rating_count[entity_id]
        )

class ItemRatingFeatureComputer:

    def __init__(self):
        self.rating_sum = {}
        self.rating_count = {}

    def update(self, movie_id, rating):

        self.rating_sum[movie_id] = (
            self.rating_sum.get(movie_id, 0) + rating
        )

        self.rating_count[movie_id] = (
            self.rating_count.get(movie_id, 0) + 1
        )

        return (
            self.rating_sum[movie_id] /
            self.rating_count[movie_id]
        )

# Retrieval:

In [33]:
class TwoTowerRetrieval(RetrievalService):

    def __init__(
        self,
        user_tower,
        item_tower,
        ann_index,
        redis,
        feature_store
    ):
        super().__init__()

        self.user_tower = user_tower
        self.item_tower = item_tower
        self.ann_index = ann_index
        self.redis = redis
        self.feature_store = feature_store


    def build_index(self, world):

        item_embeddings = {}

        for movie_id, movie in world.movie_objects.items():

            state = world.movie_states[movie_id]

            features = {
                "movie_id": movie_id,
                "genres": movie.genres,
                "average_rating": state.average_rating,
                "popularity": state.num_ratings
            }

            item_embeddings[movie_id] = (
                self.item_tower.embed(features)
            )

        self.ann_index.build(item_embeddings)


    def retrieve(self, request):

        user_id = request.user_id


        genre_preferences = self.redis.get(
            "genre_preferences",
            user_id
        )

        average_rating = self.redis.get(
            "average_rating",
            user_id
        )

        activity_level = self.redis.get(
            "activity_level",
            user_id
        )

        watch_count = self.redis.get(
            "watch_count",
            user_id
        )

        user_click_counts = self.redis.get(
            "user_click_counts",
            user_id
        )

        interactions = self.redis.get(
            "interactions",
            user_id
        )


        # Cold start bootstrap
        if genre_preferences is None:

            user_features = self.feature_store.user_features.get(
                user_id
            )

            if user_features is None:

                genre_names = [
                    "Action",
                    "Adventure",
                    "Animation",
                    "Comedy",
                    "Crime",
                    "Drama",
                    "Fantasy",
                    "Horror",
                    "Romance",
                    "Sci-Fi",
                    "Thriller",
                    "Documentary",
                    "Family",
                    "Mystery",
                    "War",
                    "Western",
                    "Music"
                ]

                weight = 1.0 / len(genre_names)

                user_features = {
                    "genre_preferences": {
                        genre: weight
                        for genre in genre_names
                    },
                    "average_rating": 3.5,
                    "activity_level": 0,
                    "watch_count": 0,
                    "user_click_counts": 0,
                    "interactions": 0
                }

                self.feature_store.write_user_features(
                    user_id,
                    user_features
                )


            genre_preferences = user_features.get(
                "genre_preferences"
            )

            average_rating = user_features.get(
                "average_rating",
                3.5
            )

            activity_level = user_features.get(
                "activity_level",
                0
            )

            watch_count = user_features.get(
                "watch_count",
                0
            )

            user_click_counts = user_features.get(
                "user_click_counts",
                0
            )

            interactions = user_features.get(
                "interactions",
                0
            )


            # Materialize complete online feature state

            for feature, value in {
                "genre_preferences": genre_preferences,
                "average_rating": average_rating,
                "activity_level": activity_level,
                "watch_count": watch_count,
                "user_click_counts": user_click_counts,
                "interactions": interactions
            }.items():

                self.redis.receive({
                    "feature": feature,
                    "user_id": user_id,
                    "value": value
                })


        # Recover missing online features from offline store

        if (
            average_rating is None or
            activity_level is None or
            watch_count is None or
            user_click_counts is None or
            interactions is None
        ):

            offline = self.feature_store.user_features.get(
                user_id,
                {}
            )


            average_rating = (
                average_rating
                if average_rating is not None
                else offline.get("average_rating", 3.5)
            )

            activity_level = (
                activity_level
                if activity_level is not None
                else offline.get("activity_level", 0)
            )

            watch_count = (
                watch_count
                if watch_count is not None
                else offline.get("watch_count", 0)
            )

            user_click_counts = (
                user_click_counts
                if user_click_counts is not None
                else offline.get("user_click_counts", 0)
            )

            interactions = (
                interactions
                if interactions is not None
                else offline.get("interactions", 0)
            )


        user_features = {
            "genre_preferences": genre_preferences,
            "average_rating": average_rating,
            "activity_level": activity_level,
            "watch_count": watch_count,
            "user_click_counts": user_click_counts,
            "interactions": interactions
        }


        user_embedding = self.user_tower.embed(
            user_features
        )


        return self.ann_index.search(
            user_embedding,
            request.k
        )

In [34]:



from abc import ABC, abstractmethod

import torch
import torch.nn as nn


class UserTower(ABC):

    @abstractmethod
    def embed(self, user_features):
        pass



class SimpleUserTower(nn.Module, UserTower):

    def __init__(self, embedding_dim=32):
        super().__init__()

        input_dim = len(genre_names) + 5

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )


    def encode_features(self, user_features):

        embedding = []

        genre_preferences = user_features["genre_preferences"]

        for genre in genre_names:
            embedding.append(
                genre_preferences.get(genre, 0.0)
            )


        embedding.append(
            user_features.get("average_rating", 0)
        )

        embedding.append(
            user_features.get("activity_level", 0)
        )

        embedding.append(
            user_features.get("watch_count", 0)
        )

        embedding.append(
            user_features.get("user_click_counts", 0)
        )

        embedding.append(
            user_features.get("interactions", 0)
        )


        return torch.tensor(
            embedding,
            dtype=torch.float32
        )


    def forward(self, x):

        return self.network(x)


    def encode(self, user_features):

        x = self.encode_features(
            user_features
        )

        return self.forward(x)


    def embed(self, user_features):

        with torch.no_grad():

            return self.encode(
                user_features
            ).tolist()



class ItemTower(ABC):

    @abstractmethod
    def embed(self, movie_features):
        pass



class SimpleItemTower(nn.Module, ItemTower):

    def __init__(
        self,
        num_movies,
        embedding_dim=32
    ):
        super().__init__()

        self.movie_embedding = nn.Embedding(
            num_movies + 1,
            16
        )

        input_dim = 16 + len(genre_names) + 2

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )


    def encode_features(self, movie_features):

        movie_id = movie_features["movie_id"]

        movie_id_embedding = self.movie_embedding(
            torch.tensor(
                movie_id,
                dtype=torch.long
            )
        )


        embedding = []

        movie_genres = set(
            movie_features["genres"]
        )

        for genre in genre_names:
            embedding.append(
                1.0 if genre in movie_genres else 0.0
            )


        embedding.append(
            movie_features["average_rating"]
        )

        embedding.append(
            movie_features["popularity"]
        )


        metadata_embedding = torch.tensor(
            embedding,
            dtype=torch.float32
        )


        return torch.cat(
            [
                movie_id_embedding,
                metadata_embedding
            ]
        )


    def forward(self, x):

        return self.network(x)


    def encode(self, movie_features):

        x = self.encode_features(
            movie_features
        )

        return self.forward(x)


    def embed(self, movie_features):

        with torch.no_grad():

            return self.encode(
                movie_features
            ).tolist()

In [35]:
class ANNIndex(ABC):

    @abstractmethod
    def build(self, item_embeddings):
        pass

    @abstractmethod
    def search(self, query_embedding, k):
        pass


class HNSWNode:

    def __init__(self, item_id, embedding, level):
        self.item_id = item_id
        self.embedding = embedding
        self.level = level

        # neighbors[level] = list of connected node ids
        self.neighbors = {
            i: []
            for i in range(level + 1)
        }

In [36]:
import random
import heapq


class HNSWIndex(ANNIndex):

    #def __init__(self, M=16, ef_construction=200):

    def __init__(
        self,
        M=16,
        ef_construction=200,
        ef_search=200
    ):

        self.M = M
        self.ef_construction = ef_construction
        self.ef_search = ef_search

        self.nodes = {}
        self.entry_point = None
        self.max_level = -1

        self.level_mult = 1 / math.log(M)

    def distance_to_node(self, query_embedding, node_id):

        return cosine_distance(
            query_embedding,
            self.nodes[node_id].embedding
        )

    def assign_level(self):
        return int(
            -math.log(random.random()) * self.level_mult
        )

    def insert(self, item_id, embedding):

        level = self.assign_level()

        node = HNSWNode(
            item_id,
            embedding,
            level
        )

        self.nodes[item_id] = node

        if self.entry_point is None:
            self.entry_point = item_id
            self.max_level = level
            return

    def cosine_distance(a, b):

        dot = sum(x * y for x, y in zip(a, b))

        norm_a = sum(x * x for x in a) ** 0.5
        norm_b = sum(y * y for y in b) ** 0.5

        return 1 - (dot / (norm_a * norm_b))


    def select_neighbors(self, candidates):

        candidates.sort(
            key=lambda x: x[1]
        )

        return candidates[:self.M]

    def insert(self, item_id, embedding):

        level = self.assign_level()

        node = HNSWNode(
            item_id,
            embedding,
            level
        )

        self.nodes[item_id] = node

        if self.entry_point is None:
            self.entry_point = item_id
            self.max_level = level
            return

        # Connect node from its highest available layer down to layer 0
        for layer in range(min(level, self.max_level), -1, -1):

            candidates = []

            for other_id, other_node in self.nodes.items():

                if other_id == item_id:
                    continue

                if layer <= other_node.level:

                    distance = cosine_distance(
                        embedding,
                        other_node.embedding
                    )

                    candidates.append(
                        (other_id, distance)
                    )

            neighbors = self.select_neighbors(candidates)

            node.neighbors[layer] = [
                neighbor_id
                for neighbor_id, _ in neighbors
            ]

            # Create bidirectional connections
            for neighbor_id in node.neighbors[layer]:

                neighbor = self.nodes[neighbor_id]

                neighbor.neighbors[layer].append(
                    item_id
                )

                self.prune_neighbors(
                    neighbor,
                    layer
                )

        # Update entry point if this node has a higher level
        if level > self.max_level:

            self.entry_point = item_id
            self.max_level = level



    def build(self, item_embeddings):

        for item_id, embedding in item_embeddings.items():

            self.insert(
                item_id,
                embedding
            )


    def prune_neighbors(self, node, layer):

        neighbors = node.neighbors[layer]

        if len(neighbors) <= self.M:
            return

        candidates = []

        for neighbor_id in neighbors:

            neighbor = self.nodes[neighbor_id]

            distance = cosine_distance(
                node.embedding,
                neighbor.embedding
            )

            candidates.append(
                (neighbor_id, distance)
            )

        selected = self.select_neighbors(candidates)

        node.neighbors[layer] = [
            neighbor_id
            for neighbor_id, _ in selected
        ]


    def search_layer_greedy(self, query_embedding, entry_point, layer):

        current = entry_point

        current_distance = self.distance_to_node(
            query_embedding,
            current
        )

        changed = True

        while changed:

            changed = False

            node = self.nodes[current]

            for neighbor_id in node.neighbors.get(layer, []):

                distance = self.distance_to_node(
                    query_embedding,
                    neighbor_id
                )

                if distance < current_distance:

                    current = neighbor_id
                    current_distance = distance
                    changed = True

        return current



    def search_layer(self, query_embedding, entry_point, ef):

        visited = set()

        # min heap: closest candidates to explore
        candidates = []

        # max heap: best results found
        results = []

        distance = self.distance_to_node(
            query_embedding,
            entry_point
        )

        heapq.heappush(
            candidates,
            (distance, entry_point)
        )

        heapq.heappush(
            results,
            (-distance, entry_point)
        )

        visited.add(entry_point)


        while candidates:

            current_distance, current_id = heapq.heappop(
                candidates
            )

            # Furthest result is better than current candidate:
            # stop expanding
            if len(results) >= ef:

                worst_distance = -results[0][0]

                if current_distance > worst_distance:
                    break


            node = self.nodes[current_id]

            for neighbor_id in node.neighbors.get(0, []):

                if neighbor_id in visited:
                    continue

                visited.add(neighbor_id)

                distance = self.distance_to_node(
                    query_embedding,
                    neighbor_id
                )

                heapq.heappush(
                    candidates,
                    (distance, neighbor_id)
                )

                heapq.heappush(
                    results,
                    (-distance, neighbor_id)
                )

                if len(results) > ef:
                    heapq.heappop(results)


        return [
            node_id
            for _, node_id in sorted(
                [
                    (-d, node_id)
                    for d, node_id in results
                ]
            )
        ]

    def search(self, query_embedding, k):

        # Start from entry point
        current = self.entry_point

        # Descend upper layers
        for layer in range(self.max_level, 0, -1):

            current = self.search_layer_greedy(
                query_embedding,
                current,
                layer
            )

        # Layer 0 efSearch
        candidates = self.search_layer(
            query_embedding,
            current,
            ef=self.ef_search
        )

        return candidates[:k]

# OFFLINE TRAINING PIPELINE:

In [37]:
class OfflineFeatureStore(Service):

    def __init__(self):
        super().__init__()

        self.name = "OfflineFeatureStore"

        self.user_features = {}
        self.item_features = {}

    def write_user_features(self, user_id, features):

        self.user_features[user_id] = features


    def write_item_features(self, movie_id, features):

        self.item_features[movie_id] = features


    def get_user_features(self, user_id):

        return self.user_features.get(
            user_id
        )


    def get_item_features(self, movie_id):

        return self.item_features.get(
            movie_id
        )

In [38]:
class OfflinePipeline:

    def __init__(
        self,
        feature_pipeline,
        dataset_generator,
        trainer,
        embedding_generator
    ):
        self.feature_pipeline = feature_pipeline
        self.dataset_generator = dataset_generator
        self.trainer = trainer
        self.embedding_generator = embedding_generator

    def run(self, events):

        self.feature_pipeline.process(events)

        for event in events:
            self.dataset_generator.add_event(event)

        training_data = (
            self.dataset_generator.build()
        )

        user_tower, item_tower = self.trainer.train(
            training_data
        )

        item_embeddings = (
            self.embedding_generator.generate(
                item_tower
            )
        )

        return {
            "user_tower": user_tower,
            "item_tower": item_tower,
            "item_embeddings": item_embeddings
        }

class OfflineFeaturePipeline:

    def __init__(self, movie_objects, feature_store):
        self.movie_objects = movie_objects
        self.feature_store = feature_store

        self.rating_features = RatingFeatureComputer()
        self.item_rating_features = ItemRatingFeatureComputer()


    def process(self, events):

        user_features = {}
        item_features = {}

        for event in events:

            if event.user_id not in user_features:
                user_features[event.user_id] = {
                    "genre_preferences": {},
                    "average_rating": 0,
                    "activity_level": 0,
                    "interactions": 0,
                    "watch_count": 0,
                    "user_click_counts": 0
                }

            user = user_features[event.user_id]


            if isinstance(event, RatingEvent):

                average_rating = self.rating_features.update(
                    event.user_id,
                    event.rating
                )

                user["average_rating"] = average_rating


            if hasattr(event, "movie_id"):

                movie = self.movie_objects[event.movie_id]

                if event.movie_id not in item_features:

                    item_features[event.movie_id] = {
                        "movie_id": event.movie_id,
                        "genres": movie.genres,
                        "average_rating": 0,
                        "popularity": 0,
                        "interactions": 0
                    }


                item = item_features[event.movie_id]

                item["interactions"] += 1

                user["interactions"] += 1
                user["activity_level"] = user["interactions"]


                if isinstance(event, WatchEvent):

                    user["watch_count"] += 1

                    item["popularity"] += 1


                    for genre in movie.genres:
                        user["genre_preferences"][genre] = (
                            user["genre_preferences"].get(genre, 0) + 1
                        )


                if isinstance(event, RecommendationClickedEvent):

                    user["user_click_counts"] += 1


                if isinstance(event, RatingEvent):

                    item_average_rating = (
                        self.item_rating_features.update(
                            event.movie_id,
                            event.rating
                        )
                    )

                    item["average_rating"] = item_average_rating


        for user_id, features in user_features.items():

            total = sum(
                features["genre_preferences"].values()
            )

            if total > 0:

                features["genre_preferences"] = {
                    genre: count / total
                    for genre, count in features["genre_preferences"].items()
                }


            self.feature_store.write_user_features(
                user_id,
                features
            )


        for movie_id, features in item_features.items():

            self.feature_store.write_item_features(
                movie_id,
                features
            )



In [39]:
class ModelRegistry:

    def __init__(self):
        self.artifacts = {}

    def save(self, name, artifact):
        self.artifacts[name] = artifact

    def load(self, name):
        return self.artifacts[name]

class OfflineArtifacts:

    def __init__(
        self,
        user_tower,
        item_tower,
        hnsw_index
    ):
        self.user_tower = user_tower
        self.item_tower = item_tower
        self.hnsw_index = hnsw_index

# TRAINING RETRIEVER

In [40]:
class TrainingDatasetGenerator:

    def __init__(
        self,
        feature_store,
        negative_samples=5
    ):
        self.feature_store = feature_store
        self.interactions = []
        self.user_movies = {}
        self.negative_samples = negative_samples


    def add_event(self, event):

        if isinstance(event, RecommendationClickedEvent):

            self.interactions.append({
                "user_id": event.user_id,
                "movie_id": event.movie_id,
                "label": 1
            })

            self.user_movies.setdefault(
                event.user_id,
                set()
            ).add(event.movie_id)


        elif isinstance(event, WatchEvent):

            self.interactions.append({
                "user_id": event.user_id,
                "movie_id": event.movie_id,
                "label": 1
            })

            self.user_movies.setdefault(
                event.user_id,
                set()
            ).add(event.movie_id)


    def add_negative_samples(self):

        all_movies = list(
            self.feature_store.item_features.keys()
        )

        negatives = []

        for interaction in self.interactions:

            user_id = interaction["user_id"]

            seen_movies = self.user_movies.get(
                user_id,
                set()
            )

            candidates = [
                movie_id
                for movie_id in all_movies
                if movie_id not in seen_movies
            ]


            for _ in range(self.negative_samples):

                if candidates:

                    negative_movie = random.choice(
                        candidates
                    )

                    negatives.append({
                        "user_id": user_id,
                        "movie_id": negative_movie,
                        "label": 0
                    })


        self.interactions.extend(
            negatives
        )


    def build(self):

        self.add_negative_samples()

        dataset = []

        for interaction in self.interactions:

            dataset.append({
                "user_features":
                    self.feature_store.get_user_features(
                        interaction["user_id"]
                    ),

                "item_features":
                    self.feature_store.get_item_features(
                        interaction["movie_id"]
                    ),

                "label":
                    interaction["label"]
            })

        self.interactions = []
        self.user_movies = {}

        return dataset


        return dataset

In [41]:




class TwoTowerTrainer:

    def __init__(
        self,
        user_tower,
        item_tower,
        learning_rate=0.001,
        epochs=5
    ):

        self.user_tower = user_tower
        self.item_tower = item_tower

        self.epochs = epochs

        self.optimizer = torch.optim.Adam(
            list(self.user_tower.parameters()) +
            list(self.item_tower.parameters()),
            lr=learning_rate
        )

        self.loss_fn = nn.BCEWithLogitsLoss()


    def train(self, training_data):

        self.user_tower.train()
        self.item_tower.train()

        for epoch in range(self.epochs):

            total_loss = 0

            for example in training_data:

                user_embedding = (
                    self.user_tower.encode(
                        example["user_features"]
                    )
                )

                item_embedding = (
                    self.item_tower.encode(
                        example["item_features"]
                    )
                )

                score = torch.dot(
                    user_embedding,
                    item_embedding
                )

                label = torch.tensor(
                    example["label"],
                    dtype=torch.float32
                )

                loss = self.loss_fn(
                    score,
                    label
                )

                self.optimizer.zero_grad()

                loss.backward()

                self.optimizer.step()

                total_loss += loss.item()


            print(
                "epoch:",
                epoch + 1,
                "loss:",
                total_loss / len(training_data)
            )


        return (
            self.user_tower,
            self.item_tower
        )


class EmbeddingGenerator:

    def __init__(self, feature_store):
        self.feature_store = feature_store

    def generate(self, item_tower):

        embeddings = {}

        for movie_id in self.feature_store.item_features:

            features = self.feature_store.get_item_features(
                movie_id
            )

            embeddings[movie_id] = (
                item_tower.embed(features)
            )

        return embeddings


# BOOTSTRAP (COLD START)

In [42]:
class BootstrapFeaturePipeline:

    def __init__(
        self,
        movie_objects,
        user_objects,
        feature_store
    ):
        self.movie_objects = movie_objects
        self.user_objects = user_objects
        self.feature_store = feature_store


    def run(self):

        # Item features
        for movie_id, movie in self.movie_objects.items():

            self.feature_store.write_item_features(
                movie_id,
                {
                    "movie_id": movie_id,
                    "genres": movie.genres,
                    "average_rating": 0,
                    "popularity": 0,
                    "interactions": 0
                }
            )

"""
        # User features
        genre_names = [
            "Action",
            "Adventure",
            "Animation",
            "Comedy",
            "Crime",
            "Drama",
            "Fantasy",
            "Horror",
            "Romance",
            "Sci-Fi",
            "Thriller",
            "Documentary",
            "Family",
            "Mystery",
            "War",
            "Western",
            "Music"
        ]

        weight = 1.0 / len(genre_names)


        existing_user_ids = random.sample(
            list(self.user_objects.keys()),
            len(self.user_objects) // 2
        )


        for user_id in existing_user_ids:

            self.feature_store.write_user_features(
                user_id,
                {
                    "genre_preferences": {
                        genre: weight
                        for genre in genre_names
                    },
                    "average_rating": 3.5,
                    "activity_level": 1,
                    "watch_count": 0,
                    "user_click_counts": 0,
                    "interactions": 0
                }
            )
   """

'\n        # User features\n        genre_names = [\n            "Action",\n            "Adventure",\n            "Animation",\n            "Comedy",\n            "Crime",\n            "Drama",\n            "Fantasy",\n            "Horror",\n            "Romance",\n            "Sci-Fi",\n            "Thriller",\n            "Documentary",\n            "Family",\n            "Mystery",\n            "War",\n            "Western",\n            "Music"\n        ]\n\n        weight = 1.0 / len(genre_names)\n\n\n        existing_user_ids = random.sample(\n            list(self.user_objects.keys()),\n            len(self.user_objects) // 2\n        )\n\n\n        for user_id in existing_user_ids:\n\n            self.feature_store.write_user_features(\n                user_id,\n                {\n                    "genre_preferences": {\n                        genre: weight\n                        for genre in genre_names\n                    },\n                    "average_rating": 3.5,\

In [43]:
class BootstrapPipeline:

    def __init__(
        self,
        feature_pipeline,
        embedding_generator
    ):
        self.feature_pipeline = feature_pipeline
        self.embedding_generator = embedding_generator

    def run(self, item_tower):

        self.feature_pipeline.run()

        item_embeddings = (
            self.embedding_generator.generate(
                item_tower
            )
        )

        hnsw_index = HNSWIndex()
        hnsw_index.build(item_embeddings)

        return {
            "item_tower": item_tower,
            "item_embeddings": item_embeddings,
            "hnsw_index": hnsw_index
        }

In [44]:
def create_new_user_features(user_id):
    return {
        "genre_preferences": {},
        "average_rating": 0,
        "activity_level": 0,
        "interactions": 0
    }

# BOOTSTRAP cold start:

In [45]:
def build_recommendation_system():

    # Feature store
    feature_store = OfflineFeatureStore()




    # Bootstrap
    bootstrap_pipeline = BootstrapPipeline(
        feature_pipeline=BootstrapFeaturePipeline(
            movie_objects,
            user_objects,
            feature_store
        ),
        embedding_generator=EmbeddingGenerator(
            feature_store
        )
    )


    bootstrap_item_tower = SimpleItemTower(
        num_movies=len(movie_objects)
    )

    bootstrap_artifacts = bootstrap_pipeline.run(
        bootstrap_item_tower
    )
    #print("USER FEATURES:")
    #print(feature_store.user_features.keys())

    #print("ITEM FEATURES:")
    #print(feature_store.item_features.keys())

    # Online components
    redis = Redis()


    retriever = TwoTowerRetrieval(
        user_tower=SimpleUserTower(),
        item_tower=bootstrap_artifacts["item_tower"],
        ann_index=bootstrap_artifacts["hnsw_index"],
        redis=redis,
        feature_store=feature_store
    )


    model_server = ModelServer(
        retriever=retriever
    )


    # Simulator
    world = World(
        user_objects=user_objects,
        movie_objects=movie_objects,
        user_states=user_states,
        movie_states=movie_states
    )


    sim = Simulator()

    sim.register(world)
    sim.register(Kafka())
    sim.register(Flink())
    sim.register(redis)
    sim.register(feature_store)

    datalake = DataLake()

    sim.register(datalake)
    sim.register(model_server)


    # Offline pipeline
    offline_feature_pipeline = OfflineFeaturePipeline(
        movie_objects,
        feature_store
    )


    training_dataset_generator = TrainingDatasetGenerator(
        feature_store
    )


    embedding_generator = EmbeddingGenerator(
        feature_store
    )


    two_tower_trainer = TwoTowerTrainer(
        user_tower=SimpleUserTower(),
        item_tower=SimpleItemTower(
            num_movies=len(movie_objects)
        )
    )


    offline_pipeline = OfflinePipeline(
        feature_pipeline=offline_feature_pipeline,
        dataset_generator=training_dataset_generator,
        trainer=two_tower_trainer,
        embedding_generator=embedding_generator
    )



    #offline_pipeline.run()

    return {
        "sim": sim,
        "world": world,
        "redis": redis,
        "datalake": datalake,
        "retriever": retriever,
        "model_server": model_server,
        "offline_pipeline": offline_pipeline,
        "feature_store": feature_store
    }

In [46]:
system = build_recommendation_system()



In [69]:
feature_store = system["feature_store"]
redis = system["redis"]

offline = feature_store.get_user_features(655)
online = redis.get("genre_preferences", 655)

print("OFFLINE:")
print(offline)

print("ONLINE:")
print(online)

OFFLINE:
{'genre_preferences': {'Action': 0.25, 'Drama': 0.25, 'Thriller': 0.25, 'War': 0.25}, 'average_rating': 3.0, 'activity_level': 3, 'interactions': 3, 'watch_count': 1, 'user_click_counts': 1}
ONLINE:
{'Action': 0.25, 'Drama': 0.25, 'Thriller': 0.25, 'War': 0.25}


In [64]:
system["sim"].run(500)

World processing
RecommendationShownEvent(timestamp=1001, user_id=320, recommendation_id=1001, movie_ids=[1599, 1085, 742, 711, 1462, 927, 1517, 136, 1626, 1466, 1524, 958, 550, 1395, 858, 741, 1183, 349, 201, 129, 1673, 1144, 396, 938, 37, 1492, 487, 1012, 1238, 55, 652, 424, 24, 77, 439, 720, 1584, 893, 160, 1403, 1213, 188, 468, 171, 1653, 1556, 457, 1564, 303, 650, 832, 1546, 17, 450, 643, 78, 754, 82, 1297, 674, 546, 18, 1254, 293, 38, 1579, 1177, 1091, 766, 639, 286, 803, 1292, 598, 888, 1405, 166, 144, 722, 633, 1657, 1028, 1658, 648, 1204, 395, 183, 1151, 458, 382, 1429, 1163, 1483, 906, 209, 213, 528, 1452, 1533, 2])
World processing
User 590 clicked 129 (score=205)
RecommendationShownEvent(timestamp=1002, user_id=590, recommendation_id=1002, movie_ids=[1599, 1085, 711, 742, 927, 1462, 1517, 1524, 136, 1466, 550, 858, 349, 1626, 741, 1673, 958, 1144, 1395, 1183, 1492, 938, 24, 129, 55, 201, 487, 1238, 439, 77, 1012, 396, 1653, 188, 720, 37, 160, 1213, 1546, 82, 652, 171, 1403,

In [49]:
user_id = 760

offline = system["feature_store"].get_user_features(user_id)
online = system["redis"].get("genre_preferences", user_id)

print("OFFLINE:")
print(offline)

print("\nONLINE:")
print(online)

OFFLINE:
{'genre_preferences': {'Action': 0.058823529411764705, 'Adventure': 0.058823529411764705, 'Animation': 0.058823529411764705, 'Comedy': 0.058823529411764705, 'Crime': 0.058823529411764705, 'Drama': 0.058823529411764705, 'Fantasy': 0.058823529411764705, 'Horror': 0.058823529411764705, 'Romance': 0.058823529411764705, 'Sci-Fi': 0.058823529411764705, 'Thriller': 0.058823529411764705, 'Documentary': 0.058823529411764705, 'Family': 0.058823529411764705, 'Mystery': 0.058823529411764705, 'War': 0.058823529411764705, 'Western': 0.058823529411764705, 'Music': 0.058823529411764705}, 'average_rating': 3.5, 'activity_level': 0, 'watch_count': 0, 'user_click_counts': 0, 'interactions': 0}

ONLINE:
{'Drama': 0.3333333333333333, 'Romance': 0.3333333333333333, 'Thriller': 0.3333333333333333}


In [50]:
print(system["redis"].get("watch_count", 760))
print(system["redis"].get("activity_level", 760))

1
2


In [67]:
user_id = 760

offline = system["feature_store"].get_user_features(user_id)

print("genre:",
      offline["genre_preferences"] ==
      system["redis"].get("genre_preferences", user_id))

print("watch_count:",
      offline["watch_count"] ==
      system["redis"].get("watch_count", user_id))

print("activity_level:",
      offline["activity_level"] ==
      system["redis"].get("activity_level", user_id))

genre: False
watch_count: False
activity_level: False


In [ ]:
user_id = 760

movies = system["retriever"].retrieve(user_id)

for movie_id in movies[:20]:
    print(
        movie_id,
        movie_objects[movie_id].genres
    )

AttributeError: 'int' object has no attribute 'user_id'

In [52]:
print("DataLake events:", len(system["datalake"].events))

for e in system["datalake"].events[-10:]:
    print(e)

DataLake events: 584
RecommendationClickedEvent(timestamp=496, user_id=568, recommendation_id=496, movie_id=311)
WatchEvent(timestamp=496, user_id=568, movie_id=311, watch_time=111.67104252164765, completion=0.2714990555524122)
RecommendationClickedEvent(timestamp=497, user_id=307, recommendation_id=497, movie_id=245)
WatchEvent(timestamp=497, user_id=307, movie_id=245, watch_time=41.4267775140796, completion=0.2141053292532553)
RecommendationClickedEvent(timestamp=498, user_id=680, recommendation_id=498, movie_id=245)
WatchEvent(timestamp=498, user_id=680, movie_id=245, watch_time=46.10760555442589, completion=0.8635347423493613)
RecommendationClickedEvent(timestamp=499, user_id=107, recommendation_id=499, movie_id=100)
WatchEvent(timestamp=499, user_id=107, movie_id=100, watch_time=36.22227478026337, completion=0.4270100728715134)
RecommendationClickedEvent(timestamp=500, user_id=900, recommendation_id=500, movie_id=100)
WatchEvent(timestamp=500, user_id=900, movie_id=100, watch_time

In [ ]:
[e for e in system["datalake"].events if e.user_id == 760]

[RecommendationClickedEvent(timestamp=4, user_id=760, recommendation_id=4, movie_id=21),
 WatchEvent(timestamp=4, user_id=760, movie_id=21, watch_time=72.69022461541181, completion=0.8272426831256716)]

In [53]:
for uid in [655,115,26]:
    print(
        uid,
        system["redis"].get(
            "genre_preferences",
            uid
        )
    )

655 {'Action': 0.25, 'Drama': 0.25, 'Thriller': 0.25, 'War': 0.25}
115 {'Action': 0.14285714285714285, 'Drama': 0.2857142857142857, 'Thriller': 0.2857142857142857, 'War': 0.14285714285714285, 'Crime': 0.14285714285714285}
26 {'Action': 0.25, 'Drama': 0.25, 'Thriller': 0.25, 'War': 0.25}


In [60]:
class Request:
    def __init__(self, user_id, k):
        self.user_id = user_id
        self.k = k


retriever = system["retriever"]

for uid in [636, 768, 159, 243]:

    request = Request(
        user_id=uid,
        k=10
    )

    result = retriever.retrieve(request)

    print("USER", uid)
    print(result)

USER 636
[1599, 1085, 742, 711, 1462, 927, 1517, 136, 1626, 1466]
USER 768
[1599, 1085, 742, 711, 1462, 927, 1517, 136, 1626, 1466]
USER 159
[1599, 1085, 742, 711, 1462, 927, 1517, 136, 1626, 1466]
USER 243
[1599, 1085, 711, 742, 927, 1462, 1517, 1524, 136, 1466]


In [ ]:
len(system["datalake"].events)

1138

In [68]:
# Rebuild offline features
offline_pipeline = system["offline_pipeline"]

offline_pipeline.feature_pipeline.process(
    system["datalake"].events
)

flink = system["sim"].get_service("Flink")
redis = system["redis"]

feature_store = (
    offline_pipeline.feature_pipeline.feature_store
)


# -------------------------
# USER average_rating
# -------------------------

user_id = next(iter(
    flink.rating_features.rating_sum
))

online = redis.get(
    "average_rating",
    user_id
)

offline = (
    feature_store
    .user_features[user_id]["average_rating"]
)

print("USER average_rating")
print("user:", user_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# USER genre_preferences
# -------------------------

user_id = next(iter(
    flink.state["genre_counts"]
))

online = redis.get(
    "genre_preferences",
    user_id
)

offline = (
    feature_store
    .user_features[user_id]["genre_preferences"]
)

print("USER genre_preferences")
print("user:", user_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# USER activity_level
# -------------------------

user_id = next(iter(
    flink.state["activity_level"]
))

online = redis.get(
    "activity_level",
    user_id
)

offline = (
    feature_store
    .user_features[user_id]["activity_level"]
)

print("USER activity_level")
print("user:", user_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# ITEM average_rating
# -------------------------

movie_id = next(iter(
    flink.item_rating_features.rating_count
))

online = redis.get(
    "item_average_rating",
    movie_id
)

offline = (
    feature_store
    .item_features[movie_id]["average_rating"]
)

print("ITEM average_rating")
print("movie:", movie_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# ITEM popularity
# -------------------------

movie_id = next(iter(
    flink.state["item_popularity"]
))

online = redis.get(
    "item_popularity",
    movie_id
)

offline = (
    feature_store
    .item_features[movie_id]["popularity"]
)

print("ITEM popularity")
print("movie:", movie_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# USER watch_count
# -------------------------

user_id = next(iter(
    flink.state["watch_count"]
))

online = redis.get(
    "watch_count",
    user_id
)

offline = (
    feature_store
    .user_features[user_id]["watch_count"]
)

print("USER watch_count")
print("user:", user_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)
print()


# -------------------------
# USER click_count
# -------------------------

user_id = next(iter(
    flink.state["user_click_counts"]
))

online = redis.get(
    "user_click_counts",
    user_id
)

offline = (
    feature_store
    .user_features[user_id]["user_click_counts"]
)

print("USER click_count")
print("user:", user_id)
print("online:", online)
print("offline:", offline)
print("match:", online == offline)

USER average_rating
user: 229
online: 5.0
offline: 5.0
match: True

USER genre_preferences
user: 115
online: {'Action': 0.2, 'Drama': 0.3, 'Thriller': 0.3, 'War': 0.1, 'Crime': 0.1}
offline: {'Action': 0.2, 'Drama': 0.3, 'Thriller': 0.3, 'War': 0.1, 'Crime': 0.1}
match: True

USER activity_level
user: 115
online: 7
offline: 7
match: True

ITEM average_rating
movie: 245
online: 3.7058823529411766
offline: 3.7058823529411766
match: True

ITEM popularity
movie: 245
online: 160
offline: 160
match: True

USER watch_count
user: 115
online: 3
offline: 3
match: True

USER click_count
user: 115
online: 3
offline: 3
match: True


In [ ]:
ratings = [
    e for e in system["datalake"].events
    if isinstance(e, RatingEvent)
]

len(ratings)

24

In [ ]:
len(system["offline_pipeline"].dataset_generator.interactions)

57

In [ ]:
len(artifacts["item_embeddings"])

1682

In [ ]:
retriever = system["model_server"].retriever

print(retriever.user_tower is artifacts["user_tower"])
print(retriever.item_tower is artifacts["item_tower"])

True
True


In [ ]:
training_data = system["offline_pipeline"].dataset_generator.build()

print("Examples:", len(training_data))
print(training_data[0])

Examples: 57
{'user_features': {'genre_preferences': {'Action': 2, 'Drama': 2, 'Mystery': 2, 'Romance': 2, 'Thriller': 2}, 'average_rating': 0, 'activity_level': 2, 'interactions': 2}, 'item_features': {'genres': ['Action', 'Drama', 'Mystery', 'Romance', 'Thriller'], 'average_rating': 0, 'popularity': 29, 'interactions': 29}, 'label': 1}


In [ ]:
len(system["redis"].store)

198

In [ ]:
feature_store = OfflineFeatureStore()

In [ ]:
len(system["retriever"].feature_store.user_features)

671

# RETRAIN OFFLINE RETRIEVER

In [55]:
def retrain(system):

    artifacts = system["offline_pipeline"].run(
        system["datalake"].events
    )

    hnsw_index = HNSWIndex()
    hnsw_index.build(
        artifacts["item_embeddings"]
    )

    system["retriever"].user_tower = artifacts["user_tower"]
    system["retriever"].item_tower = artifacts["item_tower"]
    system["retriever"].ann_index = hnsw_index

    return artifacts

In [ ]:
old = system["retriever"].ann_index.nodes[1].embedding

In [59]:
artifacts = retrain(system)

epoch: 1 loss: 0.01775319165157561
epoch: 2 loss: 0.08269520080149748
epoch: 3 loss: 0.02290257219375172
epoch: 4 loss: 0.023417776787324856
epoch: 5 loss: 0.007666059265966258


In [ ]:
for uid in [760, 307, 568]:
    print(
        uid,
        system["retriever"].retrieve(
            RecommendationRequest(uid, 20)
        )
    )

760 [1051, 1611, 83, 1420, 1071, 468, 966, 476, 805, 732, 1593, 615, 1302, 1654, 116, 1108, 721, 1362, 1001, 1394]
307 [1611, 1051, 83, 1420, 1071, 721, 615, 966, 468, 1302, 116, 1654, 1593, 476, 732, 805, 238, 1054, 869, 1108]
568 [1611, 83, 1051, 1420, 966, 468, 238, 721, 615, 116, 1071, 1654, 1593, 1302, 805, 732, 476, 1054, 869, 338]


In [ ]:
class RecommendationRequest:
    def __init__(self, user_id, k):
        self.user_id = user_id
        self.k = k


request = RecommendationRequest(
    user_id=760,
    k=20
)

recommendations = system["retriever"].retrieve(request)

print(recommendations)

[547, 1507, 603, 1230, 428, 752, 1678, 982, 761, 1459, 1674, 1003, 1407, 519, 1658, 1518, 363, 268, 1629, 175]


In [ ]:
for user_id in [760, 307, 568]:
    request = RecommendationRequest(
        user_id=user_id,
        k=20
    )

    print("\nUSER", user_id)

    print(
        "FEATURES:",
        system["redis"].get(
            "genre_preferences",
            user_id
        )
    )

    print(
        "RECS:",
        system["retriever"].retrieve(request)
    )


USER 760
FEATURES: {'Action': 0.2, 'Adventure': 0.2, 'Comedy': 0.2, 'Musical': 0.2, 'Thriller': 0.2}
RECS: [547, 1507, 603, 1230, 428, 752, 1678, 982, 761, 1459, 1674, 1003, 1407, 519, 1658, 1518, 363, 268, 1629, 175]

USER 307
FEATURES: {'Action': 0.2, 'Adventure': 0.2, 'Comedy': 0.2, 'Musical': 0.2, 'Thriller': 0.2}
RECS: [547, 1507, 603, 1230, 428, 752, 1678, 982, 761, 1459, 1674, 1003, 1407, 519, 1658, 1518, 363, 268, 1629, 175]

USER 568
FEATURES: {'Comedy': 0.3333333333333333, 'Drama': 0.3333333333333333, 'Thriller': 0.3333333333333333}
RECS: [547, 1507, 1230, 603, 428, 752, 1678, 982, 1459, 761, 1674, 1407, 1658, 1003, 519, 1518, 363, 268, 175, 1629]


In [56]:
for user_id in [760, 307, 568]:

    genre_preferences = system["redis"].get(
        "genre_preferences",
        user_id
    )

    request = RecommendationRequest(
        user_id=user_id,
        k=20
    )

    print("\nUSER", user_id)
    print("FEATURES:")
    print(genre_preferences)

NameError: name 'RecommendationRequest' is not defined

In [ ]:
user_id = 760

offline = system["feature_store"].get_user_features(user_id)
online = system["redis"].get("genre_preferences", user_id)

print(
    offline["genre_preferences"] == online
)

True


In [ ]:
for k, v in system["redis"].store.items():
    if k[0] == "genre_preferences" and v is None:
        print(k)

In [ ]:
for uid in [636, 768, 159, 243]:
    print(uid, system["redis"].get("genre_preferences", uid))

636 {'Action': 0.058823529411764705, 'Adventure': 0.058823529411764705, 'Animation': 0.058823529411764705, 'Comedy': 0.058823529411764705, 'Crime': 0.058823529411764705, 'Drama': 0.058823529411764705, 'Fantasy': 0.058823529411764705, 'Horror': 0.058823529411764705, 'Romance': 0.058823529411764705, 'Sci-Fi': 0.058823529411764705, 'Thriller': 0.058823529411764705, 'Documentary': 0.058823529411764705, 'Family': 0.058823529411764705, 'Mystery': 0.058823529411764705, 'War': 0.058823529411764705, 'Western': 0.058823529411764705, 'Music': 0.058823529411764705}
768 {'Comedy': 0.3333333333333333, 'Drama': 0.3333333333333333, 'Romance': 0.3333333333333333}
159 {'Action': 0.2, 'Adventure': 0.2, 'Comedy': 0.2, 'Musical': 0.2, 'Thriller': 0.2}
243 {'Drama': 0.3333333333333333, 'Romance': 0.3333333333333333, 'War': 0.3333333333333333}


In [ ]:
for uid in [636, 768, 159, 243]:
    print("\nUSER", uid)
    print("genre_preferences :", system["redis"].get("genre_preferences", uid))
    print("average_rating    :", system["redis"].get("average_rating", uid))
    print("activity_level    :", system["redis"].get("activity_level", uid))
    print("watch_count       :", system["redis"].get("watch_count", uid))
    print("user_click_counts :", system["redis"].get("user_click_counts", uid))


USER 636
genre_preferences : {'Action': 0.25, 'Drama': 0.25, 'Thriller': 0.25, 'War': 0.25}
average_rating    : 3.5
activity_level    : 2
watch_count       : 1
user_click_counts : 1

USER 768
genre_preferences : {'Crime': 0.14285714285714285, 'Drama': 0.2857142857142857, 'Romance': 0.2857142857142857, 'Thriller': 0.14285714285714285, 'Comedy': 0.14285714285714285}
average_rating    : 1.0
activity_level    : 5
watch_count       : 2
user_click_counts : 2

USER 159
genre_preferences : {'Action': 0.058823529411764705, 'Adventure': 0.058823529411764705, 'Animation': 0.058823529411764705, 'Comedy': 0.058823529411764705, 'Crime': 0.058823529411764705, 'Drama': 0.058823529411764705, 'Fantasy': 0.058823529411764705, 'Horror': 0.058823529411764705, 'Romance': 0.058823529411764705, 'Sci-Fi': 0.058823529411764705, 'Thriller': 0.058823529411764705, 'Documentary': 0.058823529411764705, 'Family': 0.058823529411764705, 'Mystery': 0.058823529411764705, 'War': 0.058823529411764705, 'Western': 0.0588235

In [ ]:
retriever = system["retriever"]

print("Retriever user tower:")
print(retriever.user_tower)

print("\nRetriever item tower:")
print(retriever.item_tower)

# compare embedding output before retrieval
for uid in [636,768,159,243]:
    features = {
        "genre_preferences": system["redis"].get("genre_preferences", uid),
        "average_rating": system["redis"].get("average_rating", uid),
        "activity_level": system["redis"].get("activity_level", uid),
        "watch_count": system["redis"].get("watch_count", uid),
        "user_click_counts": system["redis"].get("user_click_counts", uid)
    }

    emb = retriever.user_tower.embed(features)

    print(uid, emb[:5])

Retriever user tower:
SimpleUserTower(
  (network): Sequential(
    (0): Linear(in_features=24, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
)

Retriever item tower:
SimpleItemTower(
  (movie_embedding): Embedding(1683, 16)
  (network): Sequential(
    (0): Linear(in_features=37, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
)


AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
retriever = system["retriever"]

# rebuild ANN after training
retriever.build_index(system["world"])

print("ANN size:", len(retriever.ann_index.nodes))

# inspect retrieval directly (no fake RecommendationRequest class)
for user_id in [636, 768, 159, 243]:

    print("\nUSER", user_id)

    features = {
        "genre_preferences": system["redis"].get("genre_preferences", user_id),
        "average_rating": system["redis"].get("average_rating", user_id),
        "activity_level": system["redis"].get("activity_level", user_id),
        "watch_count": system["redis"].get("watch_count", user_id),
        "user_click_counts": system["redis"].get("user_click_counts", user_id)
    }

    embedding = retriever.user_tower.embed(features)

    print(
        retriever.ann_index.search(
            embedding,
            10
        )
    )

ANN size: 1682

USER 636
[1187, 1421, 1537, 1263, 1193, 1335, 1211, 1143, 359, 555]

USER 768
[1537, 1263, 1421, 1187, 1335, 1193, 1211, 359, 1143, 555]

USER 159
[1537, 1421, 1263, 1187, 1335, 1193, 1211, 359, 1143, 555]

USER 243
[1421, 1537, 1187, 1263, 1193, 1335, 1211, 1143, 359, 555]


In [ ]:
for uid in [636,768,159,243]:
    emb = system["retriever"].user_tower.embed({
        "genre_preferences": system["redis"].get("genre_preferences", uid),
        "average_rating": system["redis"].get("average_rating", uid),
        "activity_level": system["redis"].get("activity_level", uid),
        "watch_count": system["redis"].get("watch_count", uid),
        "user_click_counts": system["redis"].get("user_click_counts", uid),
    })
    print(uid, emb[:5])

636 [0.2079194337129593, 0.009090229868888855, -0.05558176711201668, -0.07482879608869553, 0.1909276396036148]
768 [0.24475958943367004, -0.04319785535335541, 0.002627480775117874, -0.030317306518554688, 0.23802240192890167]
159 [0.37920936942100525, 0.029538795351982117, 0.039182718843221664, -0.15961305797100067, 0.25313103199005127]
243 [0.5261185169219971, 0.06807726621627808, 0.07971695065498352, -0.3333134055137634, 0.25512903928756714]


In [ ]:
for uid in [636,768,159,243]:
    print(
        uid,
        system["redis"].get("watch_count", uid),
        system["redis"].get("user_click_counts", uid),
        system["redis"].get("activity_level", uid),
        system["redis"].get("genre_preferences", uid)
    )

636 1 1 2 {'Action': 0.5, 'Drama': 0.5}
768 1 1 2 {'Crime': 0.25, 'Drama': 0.25, 'Romance': 0.25, 'Thriller': 0.25}
159 1 2 3 {'Action': 0.25, 'Mystery': 0.25, 'Romance': 0.25, 'Thriller': 0.25}
243 2 2 4 {'Comedy': 0.4, 'Drama': 0.4, 'War': 0.2}


In [ ]:
retriever = system["retriever"]

for uid in [636,768,159,243]:
    features = {
        "genre_preferences": system["redis"].get("genre_preferences", uid),
        "average_rating": system["redis"].get("average_rating", uid),
        "activity_level": system["redis"].get("activity_level", uid),
        "watch_count": system["redis"].get("watch_count", uid),
        "user_click_counts": system["redis"].get("user_click_counts", uid)
    }

    emb = retriever.user_tower.embed(features)

    print(uid, retriever.ann_index.search(emb, 10))

636 [1187, 1421, 1537, 1263, 1193, 1335, 1211, 1143, 359, 555]
768 [1537, 1263, 1421, 1187, 1335, 1193, 1211, 359, 1143, 555]
159 [1537, 1421, 1263, 1187, 1335, 1193, 1211, 359, 1143, 555]
243 [1421, 1537, 1187, 1263, 1193, 1335, 1211, 1143, 359, 555]


In [ ]:
retriever = system["retriever"]

ids = [1187, 1421, 1537, 1263, 1193, 1335, 1211, 1143, 359, 555]

for movie_id in ids:
    emb = retriever.ann_index.nodes[movie_id].embedding
    print(movie_id, emb[:5])

1187 [0.05522346496582031, 0.1106056198477745, 0.009557473473250866, 0.15141798555850983, 0.29774826765060425]
1421 [0.08152087032794952, 0.0710097998380661, 0.15961968898773193, 0.2338767647743225, 0.3635559678077698]
1537 [0.06626388430595398, -0.07208044826984406, 0.39573660492897034, 0.4113352596759796, 0.2584438621997833]
1263 [-0.08800536394119263, -0.16674458980560303, -0.08805899322032928, 0.16423343122005463, 0.31606873869895935]
1193 [-0.06743869930505753, 0.13584229350090027, 0.11057689785957336, 0.08463256061077118, 0.16773216426372528]
1335 [-0.04223392903804779, 0.13353624939918518, 0.1346394568681717, 0.1671075075864792, 0.22043411433696747]
1211 [-0.06445997953414917, 0.10891631245613098, 0.12939095497131348, 0.1310230791568756, 0.13258428871631622]
1143 [0.09049516916275024, 0.15030361711978912, -0.05586652085185051, 0.15196487307548523, 0.23207348585128784]
359 [-0.38889628648757935, -0.08325931429862976, 0.18834292888641357, 0.06936980783939362, 0.33835092186927795]


In [ ]:
retriever = system["retriever"]

node = retriever.ann_index.nodes[1187]

print(vars(node))

{'item_id': 1187, 'embedding': [0.05522346496582031, 0.1106056198477745, 0.009557473473250866, 0.15141798555850983, 0.29774826765060425, 0.010820209980010986, -0.030189726501703262, -0.36450591683387756, 0.05797787010669708, -0.2342795431613922, -0.15224076807498932, -0.039245493710041046, -0.1673869788646698, -0.1822580099105835, -0.5673842430114746, 0.1485777199268341, -0.10680939257144928, 0.1433791220188141, -0.05688200891017914, 0.0361170656979084, 0.15322938561439514, 0.22037719190120697, -0.14774933457374573, -0.049220696091651917, -0.32570505142211914, -0.08925039321184158, 0.07700744271278381, -0.15031245350837708, 0.014068298041820526, 0.22239179909229279, 0.0786408931016922, 0.21029788255691528], 'level': 0, 'neighbors': {0: [1143, 1421, 1421, 1335, 1335, 1448, 1448, 1427, 1375, 996, 555, 1190, 617, 1003, 1512, 1189]}}


In [ ]:
for movie_id in [207, 313, 1483, 161, 849]:
    print(
        movie_id,
        system["retriever"].ann_index.nodes[movie_id].embedding[:5]
    )

207 [2.87750244140625, -4.053192615509033, -2.6394522190093994, 5.1891069412231445, -3.906578540802002]
313 [1.7738231420516968, -2.914794683456421, -2.099562644958496, 3.9855175018310547, -2.6254732608795166]
1483 [2.15090012550354, -4.105091571807861, -1.9750903844833374, 4.459883213043213, -3.6782031059265137]
161 [1.5921896696090698, -2.920973300933838, -1.9988209009170532, 2.910365104675293, -2.264463424682617]
849 [2.197941303253174, -3.929520606994629, -2.3823182582855225, 4.698526382446289, -3.7395501136779785]


In [ ]:
for user_id in [1, 2, 3, 4]:

    request = TestRequest(
        user_id,
        10
    )

    print(
        user_id,
        system["retriever"].retrieve(request)
    )

1 [1187, 1421, 1143, 1193, 1335, 1211, 1537, 1263, 555, 1448]
2 [732, 603, 144, 288, 328, 318, 300, 289, 98, 204]
3 [1187, 1421, 1143, 1193, 1335, 1211, 1537, 1263, 555, 1448]
4 [732, 603, 144, 288, 328, 318, 300, 289, 98, 204]


In [ ]:
results = system["retriever"].retrieve(request)

print(results[:10])

[1462, 280, 1461, 788, 718, 64, 1280, 775, 126, 400]


In [ ]:
retriever = system["retriever"]

print("Item tower:", type(retriever.item_tower).__name__)
print("ANN attrs:", vars(retriever.ann_index).keys())

if hasattr(system["redis"], "store"):
    print("Redis entries:", len(system["redis"].store))
else:
    print("Redis:", vars(system["redis"]).keys())

class Req:
    def __init__(self, user_id, k):
        self.user_id = user_id
        self.k = k

request = Req(
    user_id=263,
    k=10
)

print("Recommendations:")
print(retriever.retrieve(request))

Item tower: SimpleItemTower
ANN attrs: dict_keys(['M', 'ef_construction', 'ef_search', 'nodes', 'entry_point', 'max_level', 'level_mult'])
Redis entries: 2689
Recommendations:
[1462, 280, 788, 1461, 64, 1049, 162, 1280, 718, 1209]


In [ ]:
for user_id in [263, 786, 378, 172]:

    features = system["retriever"].feature_store.get_user_features(
        user_id
    )

    print(
        user_id,
        system["retriever"].user_tower.embed(features)[:5]
    )

263 [0.04084247350692749, -0.2987986207008362, -0.44882360100746155, 0.5872523784637451, 0.7093454599380493]
786 [0.040821373462677, -0.3189462125301361, -0.3952297270298004, 0.5981670618057251, 0.6993447542190552]
378 [0.04084247350692749, -0.2987986207008362, -0.44882360100746155, 0.5872523784637451, 0.7093454599380493]
172 [0.040821373462677, -0.3189462125301361, -0.3952297270298004, 0.5981670618057251, 0.6993447542190552]


In [ ]:
for user_id in [263, 786, 378, 172]:
    print(
        user_id,
        system["retriever"].feature_store.get_user_features(user_id)
    )

263 {'genre_preferences': {'Action': 0.058823529411764705, 'Adventure': 0.058823529411764705, 'Animation': 0.058823529411764705, 'Comedy': 0.058823529411764705, 'Crime': 0.058823529411764705, 'Drama': 0.058823529411764705, 'Fantasy': 0.058823529411764705, 'Horror': 0.058823529411764705, 'Romance': 0.058823529411764705, 'Sci-Fi': 0.058823529411764705, 'Thriller': 0.058823529411764705, 'Documentary': 0.058823529411764705, 'Family': 0.058823529411764705, 'Mystery': 0.058823529411764705, 'War': 0.058823529411764705, 'Western': 0.058823529411764705, 'Music': 0.058823529411764705}, 'average_rating': 3.5, 'activity_level': 1, 'interactions': 0}
786 {'genre_preferences': {'Action': 0.058823529411764705, 'Adventure': 0.058823529411764705, 'Animation': 0.058823529411764705, 'Comedy': 0.058823529411764705, 'Crime': 0.058823529411764705, 'Drama': 0.058823529411764705, 'Fantasy': 0.058823529411764705, 'Horror': 0.058823529411764705, 'Romance': 0.058823529411764705, 'Sci-Fi': 0.058823529411764705, '

In [ ]:
class TestRequest:
    def __init__(self, user_id, k):
        self.user_id = user_id
        self.k = k


for user_id in [1, 2, 3, 4]:

    request = TestRequest(
        user_id,
        10
    )

    print(
        user_id,
        system["retriever"].retrieve(request)
    )

1 [195, 298, 945, 1069, 1129, 510, 1244, 353, 802, 245]
2 [195, 298, 945, 1069, 1129, 510, 1244, 353, 802, 245]
3 [195, 298, 945, 1069, 1129, 510, 1244, 353, 802, 245]
4 [195, 298, 945, 1069, 1129, 510, 1244, 353, 802, 245]


In [ ]:
for movie_id in [207, 313, 1483, 161, 849]:
    print(
        movie_id,
        system["retriever"].ann_index.nodes[movie_id].embedding[:5]
    )

207 [4.171679973602295, -0.13351716101169586, 4.005340099334717, 2.9775350093841553, -3.8984479904174805]
313 [4.171679973602295, -0.13351716101169586, 4.005340099334717, 2.9775350093841553, -3.8984479904174805]
1483 [4.171679973602295, -0.13351716101169586, 4.005340099334717, 2.9775350093841553, -3.8984479904174805]
161 [3.4082813262939453, -0.06150777265429497, 3.105839252471924, 2.0083727836608887, -3.0044493675231934]
849 [3.4082813262939453, -0.06150777265429497, 3.105839252471924, 2.0083727836608887, -3.0044493675231934]


In [ ]:
print(system["retriever"].user_tower is artifacts["user_tower"])
print(system["retriever"].item_tower is artifacts["item_tower"])

True
True


In [ ]:
len(artifacts)

3

In [ ]:
artifacts.keys()

dict_keys(['user_tower', 'item_tower', 'item_embeddings'])

In [ ]:
Event logs
    |
    v
TrainingDatasetGenerator
    |
    v
TwoTowerTrainer
    |
    +--> UserTower
    |
    +--> ItemTower
    |
    v
EmbeddingGenerator
    |
    v
HNSWIndexBuilder

In [ ]:
World
  |
  | generates raw behavior
  v
Events
  |
  +----------------+
  |                |
  v                v
Flink            DataLake
  |                |
  v                v
Redis       Offline Feature Pipeline
                    |
                    v
             TrainingDatasetGenerator
                    |
                    v
             TwoTowerTrainer

World owns simulation of users/items/events.
Flink owns online feature computation.
Redis owns online feature serving.
DataLake owns historical raw data.
Offline feature computation owns training features.
TrainingDatasetGenerator should consume those offline features.
TwoTowerTrainer should never read World.

# Testing retrivial

In [ ]:
item_embeddings = {}

item_tower = SimpleItemTower()

for movie_id, movie in movie_objects.items():

    state = movie_states[movie_id]

    features = {
        "genres": movie.genres,
        "average_rating": state.average_rating,
        "popularity": state.num_ratings,
    }

    item_embeddings[movie_id] = item_tower.embed(features)

In [ ]:
hnsw_index = HNSWIndex()

hnsw_index.build(item_embeddings)

In [ ]:
world = World(
    user_objects=user_objects,
    movie_objects=movie_objects,
    user_states=user_states,
    movie_states=movie_states
)

sim = Simulator()

sim.register(world)
sim.register(Kafka())
sim.register(Flink())
sim.register(Redis())



#sim.register(ModelServer())

retriever = TwoTowerRetrieval(
    user_tower=SimpleUserTower(),
    item_tower=item_tower,
    ann_index=hnsw_index,
    redis=redis
)

model_server = ModelServer(
    retriever=retriever
)

sim.register(model_server)

In [ ]:
item_embeddings = {}

item_tower = SimpleItemTower()

for movie_id, movie in movie_objects.items():

    state = movie_states[movie_id]

    features = {
        "genres": movie.genres,
        "average_rating": state.average_rating,
        "popularity": state.num_ratings,
    }

    item_embeddings[movie_id] = item_tower.embed(features)

In [ ]:
hnsw_index = HNSWIndex()

hnsw_index.build(item_embeddings)

In [ ]:
print(hnsw_index.entry_point)
print(hnsw_index.max_level)
print(len(hnsw_index.nodes))

184
2
1682


In [ ]:
for node in hnsw_index.nodes.values():
    for layer, neighbors in node.neighbors.items():
        assert len(neighbors) <= hnsw_index.M

print("HNSW graph valid")

from collections import Counter

levels = Counter(
    node.level
    for node in hnsw_index.nodes.values()
)

print(levels)

HNSW graph valid
Counter({0: 1592, 1: 84, 2: 5, 3: 1})


In [ ]:
# for testing, fill redis with features

redis = sim.get_service("Redis")

for user_id, state in user_states.items():
    redis.set(
        "genre_preferences",
        user_id,
        state.genre_preferences
    )

    redis.set(
        "average_rating",
        user_id,
        state.average_rating
    )

    redis.set(
        "activity_level",
        user_id,
        state.activity_level
    )

In [ ]:
user_features = {
    "genre_preferences": redis.store.get(
        ("genre_preferences", 1)
    ),
    "average_rating": redis.store.get(
        ("average_rating", 1)
    ),
    "activity_level": redis.store.get(
        ("activity_level", 1)
    )
}

user_embedding = SimpleUserTower().embed(user_features)

hnsw_results = hnsw_index.search(
    user_embedding,
    5
)

brute_results = []

scores = []

for movie_id, node in hnsw_index.nodes.items():

    distance = cosine_distance(
        user_embedding,
        node.embedding
    )

    scores.append(
        (movie_id, distance)
    )

scores.sort(key=lambda x: x[1])

brute_results = [
    movie_id
    for movie_id, _ in scores[:5]
]

print("HNSW:", hnsw_results)
print("Brute:", brute_results)

HNSW: [1339, 1557, 1569, 1626, 1308]
Brute: [1559, 1339, 1557, 1569, 1626]


In [ ]:
for name, results in [
    ("HNSW", hnsw_results),
    ("Brute", brute_results)
]:
    print(name)

    for movie_id in results:
        d = cosine_distance(
            user_embedding,
            hnsw_index.nodes[movie_id].embedding
        )

        print(movie_id, d)

HNSW
1339 0.3168331456445749
1557 0.3168331456445749
1569 0.3168331456445749
1626 0.3168331456445749
1308 0.3447187910446726
Brute
1559 0.315265324099557
1339 0.3168331456445749
1557 0.3168331456445749
1569 0.3168331456445749
1626 0.3168331456445749


# Test the pipeline:

In [ ]:
item_embeddings = {}

item_tower = SimpleItemTower()

for movie_id, movie in movie_objects.items():

    state = movie_states[movie_id]

    features = {
        "genres": movie.genres,
        "average_rating": state.average_rating,
        "popularity": state.num_ratings,
    }

    item_embeddings[movie_id] = item_tower.embed(features)

In [ ]:
hnsw_index = HNSWIndex()

hnsw_index.build(item_embeddings)

In [ ]:
sim = Simulator()

world = World(
    user_objects=user_objects,
    movie_objects=movie_objects,
    user_states=user_states,
    movie_states=movie_states
)

redis = Redis()

#retriever.build_index(world)

item_tower = SimpleItemTower()

retriever = TwoTowerRetrieval(
    user_tower=SimpleUserTower(),
    item_tower=item_tower,
    ann_index=hnsw_index,
    redis=redis
)

#retriever = TwoTowerRetrieval(
  #  user_tower=artifacts["user_tower"],
  # item_tower=artifacts["item_tower"],
  #  ann_index=hnsw_index,
  #  redis=redis
#)

#retriever.build_index(world)

model_server = ModelServer(
    retriever=retriever
)

sim.register(world)
sim.register(Kafka())
sim.register(Flink())
sim.register(redis)
sim.register(model_server)

datalake = DataLake()
sim.register(datalake)

for state in user_states.values():

    redis.receive({
        "feature": "genre_preferences",
        "user_id": state.user_id,
        "value": state.genre_preferences
    })

    redis.receive({
        "feature": "average_rating",
        "user_id": state.user_id,
        "value": state.average_rating
    })

    redis.receive({
        "feature": "activity_level",
        "user_id": state.user_id,
        "value": state.activity_level
    })

sim.run(50)

World processing
User 655 clicked 6 (score=1223)
RecommendationShownEvent(timestamp=1, user_id=655, recommendation_id=1, movie_ids=[6, 9, 15, 18, 19, 30, 37, 46, 52, 57, 58, 59, 60, 61, 64, 86, 87, 193])
RecommendationClickedEvent(timestamp=1, user_id=655, recommendation_id=1, movie_id=6)
WatchEvent(timestamp=1, user_id=655, movie_id=6, watch_time=114.48169884246649, completion=0.20311958243569547)
Flink feature: 655 1
Flink feature: 655 watch_count 1
World processing
User 115 clicked 6 (score=166)
RecommendationShownEvent(timestamp=2, user_id=115, recommendation_id=2, movie_ids=[6, 9, 15, 18, 19, 30, 37, 46, 52, 57, 58, 59, 60, 61, 64, 86, 87, 193])
RecommendationClickedEvent(timestamp=2, user_id=115, recommendation_id=2, movie_id=6)
Flink feature: 115 1
World processing
User 26 clicked 6 (score=135)
RecommendationShownEvent(timestamp=3, user_id=26, recommendation_id=3, movie_ids=[6, 9, 15, 18, 19, 30, 37, 46, 52, 57, 58, 59, 60, 61, 64, 86, 87, 193])
RecommendationClickedEvent(timest

In [ ]:
datalake = sim.get_service("DataLake")

print(len(datalake.events))

51


In [ ]:
item_embeddings = {}

item_tower = SimpleItemTower()

for movie_id, movie in movie_objects.items():

    features = {
        "genres": movie.genres,
        "average_rating": 0,
        "popularity": 0,
    }

    item_embeddings[movie_id] = item_tower.embed(features)

In [ ]:
hnsw_index = HNSWIndex()
hnsw_index.build(item_embeddings)

In [ ]:
sim = Simulator()

world = World(
    user_objects=user_objects,
    movie_objects=movie_objects,
    user_states=user_states,
    movie_states=movie_states
)

redis = Redis()

#retriever.build_index(world)

item_tower = SimpleItemTower()

#retriever = TwoTowerRetrieval(
  #  user_tower=SimpleUserTower(),
  #  item_tower=item_tower,
   # ann_index=hnsw_index,
   # redis=redis
#)

retriever = TwoTowerRetrieval(
    user_tower=artifacts["user_tower"],
    item_tower=artifacts["item_tower"],
    ann_index=hnsw_index,
    redis=redis
)

#retriever.build_index(world)

model_server = ModelServer(
    retriever=retriever
)

sim.register(world)
sim.register(Kafka())
sim.register(Flink())
sim.register(redis)
sim.register(model_server)

datalake = DataLake()
sim.register(datalake)

for state in user_states.values():

    redis.receive({
        "feature": "genre_preferences",
        "user_id": state.user_id,
        "value": state.genre_preferences
    })

    redis.receive({
        "feature": "average_rating",
        "user_id": state.user_id,
        "value": state.average_rating
    })

    redis.receive({
        "feature": "activity_level",
        "user_id": state.user_id,
        "value": state.activity_level
    })

sim.run(50)

NameError: name 'artifacts' is not defined